In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/appledisease/Test/Blotch_Apple/149.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/143.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/131.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/130.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/120.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/124.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/123.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/137.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/127.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/140.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/145.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/135.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/121.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/126.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/142.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/148.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/138.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/146.jpg
/kaggle/input/appledisease/Test/Blotch_Apple/1

In [67]:
import numpy as np
import pandas as pd
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image


In [68]:
train_dir = "/kaggle/input/appledisease/Train"
test_dir = "/kaggle/input/appledisease/Test"

In [69]:
img_size = (224, 224) 
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [70]:
val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input, validation_split=0.2)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [71]:
train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True
)


Found 307 images belonging to 4 classes.


In [72]:
val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Found 75 images belonging to 4 classes.


In [73]:
test_gen = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

Found 120 images belonging to 4 classes.


In [74]:
for subdir, _, files in os.walk(train_dir):
    for file in files:
        try:
            img_path = os.path.join(subdir, file)
            img = Image.open(img_path)
            img.verify()
        except Exception as e:
            print(f"Corrupted or unreadable file: {img_path} — {e}")


In [75]:
labels_from_gen = train_gen.classes
class_weight = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels_from_gen),
    y=labels_from_gen
)
class_weight = dict(enumerate(class_weight))


In [76]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_reduce = ReduceLROnPlateau(monitor='val_loss', patience=3, verbose=1, factor=0.5)

In [77]:
base_model = ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

In [78]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(train_gen.num_classes, activation='softmax')
])

In [79]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [80]:
print("Phase 1: Training with frozen ResNet base")
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    class_weight=class_weight,
    callbacks=[early_stop, lr_reduce]
)

Phase 1: Training with frozen ResNet base


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.2410 - loss: 1.8823 - val_accuracy: 0.4400 - val_loss: 1.3572 - learning_rate: 1.0000e-04
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 495ms/step - accuracy: 0.3183 - loss: 1.4971 - val_accuracy: 0.5200 - val_loss: 1.1567 - learning_rate: 1.0000e-04
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 476ms/step - accuracy: 0.4010 - loss: 1.3288 - val_accuracy: 0.6533 - val_loss: 0.9834 - learning_rate: 1.0000e-04
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 487ms/step - accuracy: 0.5143 - loss: 1.1078 - val_accuracy: 0.7067 - val_loss: 0.8533 - learning_rate: 1.0000e-04
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 482ms/step - accuracy: 0.5787 - loss: 1.0679 - val_accuracy: 0.7600 - val_loss: 0.7663 - learning_rate: 1.0000e-04
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 492ms/step - accuracy: 0.6560 - loss: 0.8686 - val_accuracy: 0.8133 - val_loss: 0.6994 - learning_rate: 1.0000e-04
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 477ms/step - accuracy: 0.

In [81]:
print("Phase 2: Fine-tuning the ResNet model")
base_model.trainable = True
for layer in base_model.layers[:100]:  
    layer.trainable = False

Phase 2: Fine-tuning the ResNet model


In [82]:
model.compile(optimizer=tf.keras.optimizers.Adam(3e-6),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [83]:
early_stop2 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
lr_reduce2 = ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5)

In [84]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weight,
    callbacks=[early_stop2, lr_reduce2]
)

Epoch 1/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.7207 - loss: 0.7116 - val_accuracy: 0.9067 - val_loss: 0.5026 - learning_rate: 3.0000e-06
Epoch 2/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 494ms/step - accuracy: 0.7480 - loss: 0.6806 - val_accuracy: 0.9067 - val_loss: 0.4947 - learning_rate: 3.0000e-06
Epoch 3/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 479ms/step - accuracy: 0.7437 - loss: 0.6593 - val_accuracy: 0.8800 - val_loss: 0.4873 - learning_rate: 3.0000e-06
Epoch 4/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 496ms/step - accuracy: 0.6956 - loss: 0.7253 - val_accuracy: 0.8933 - val_loss: 0.4807 - learning_rate: 3.0000e-06
Epoch 5/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 494ms/step - accuracy: 0.7804 - loss: 0.6336 - val_accuracy: 0.8800 - val_loss: 0.4752 - learning_rate: 3.0000e-06
Epoch 6/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 481ms/step - accuracy: 0.7942 - loss: 0.6167 - val_accuracy: 0.8800 - val_loss: 0.4690 - learning_rate: 3.0000e-06
Epoch 7/15
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 514ms/step - accuracy: 0.

In [85]:
best_val_acc = max(history2.history['val_accuracy'])
best_train_acc = max(history2.history['accuracy'])

print(f"\nBest Training Accuracy: {best_train_acc:.4f}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")



Best Training Accuracy: 0.8436
Best Validation Accuracy: 0.9067


In [86]:
print("Best Validation Accuracy:", max(history2.history['val_accuracy']))


Best Validation Accuracy: 0.9066666960716248
